# Pipeline Architecture — CrewAI

**Pattern:** ETL agents — each agent is a pure transform

```
extractor → transformer → loader → TravelRankingReport
```

Agents are data transforms, not decision-makers. Each has ONE tool that does one thing.

In [ ]:
import os, json
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)
RAW = ["Tokyo|Clear|18|Low|10|22:30 JST", "Paris|Partly Cloudy|16|Low|8|15:30 CET", "Bangalore|Rainy|25|Medium|6|20:00 IST"]
print("✓ Ready")

In [ ]:
@tool("Raw Data Parser")
def parse_raw_data(raw_records: str) -> str:
    """Parse pipe-delimited city records into JSON. Input: newline-separated records."""
    records = []
    for line in raw_records.strip().split("\n"):
        p = line.split("|")
        if len(p)==6:
            c,w,t,sl,ss,tm = p
            records.append({"city":c,"weather":w,"temp_c":int(t),"safety_level":sl,"safety_score":int(ss),"local_time":tm})
    return json.dumps(records)

@tool("Score Calculator")
def calculate_scores(json_records: str) -> str:
    """Add weather_score, combined_score, rank. Input: JSON string."""
    records = json.loads(json_records)
    sm = {"Clear":9,"Partly Cloudy":7,"Rainy":6,"Cloudy":5}
    for r in records:
        r["weather_score"] = sm.get(r["weather"],5)
        r["combined_score"] = round((r["weather_score"]+r["safety_score"])/2,1)
    records.sort(key=lambda x:x["combined_score"],reverse=True)
    for i,r in enumerate(records,1): r["rank"] = i
    return json.dumps(records)

class TravelRankingReport(BaseModel):
    rankings: List[dict]; summary: str; top_city: str

print("Tools + schema ready")

In [ ]:
extractor   = Agent(role="Data Extractor",goal="Parse raw records into JSON.",backstory="You parse pipe-delimited data.",tools=[parse_raw_data],llm=gemini,verbose=False)
transformer = Agent(role="Data Transformer",goal="Score and rank records.",backstory="You add computed fields.",tools=[calculate_scores],llm=gemini,verbose=False)
loader      = Agent(role="Report Generator",goal="Format data into a report.",backstory="You write clear data reports.",tools=[],llm=gemini,verbose=False)

raw_str = "\n".join(RAW)
t1 = Task(description=f"Parse these records with parse_raw_data:\n{raw_str}", expected_output="JSON array.", agent=extractor)
t2 = Task(description="Enrich with calculate_scores.", expected_output="Enriched JSON.", agent=transformer, context=[t1])
t3 = Task(description="Create TravelRankingReport.", expected_output="Complete report.", agent=loader, context=[t2], output_pydantic=TravelRankingReport)

result = Crew(agents=[extractor,transformer,loader],tasks=[t1,t2,t3],process=Process.sequential,verbose=False).kickoff()
rpt: TravelRankingReport = result.pydantic
if rpt:
    for r in rpt.rankings: print(f"  {r.get('rank')}. {r.get('city')} — {r.get('combined_score')}")
    print(f"\nTop: {rpt.top_city}\n{rpt.summary[:100]}...")
else: print(result.raw[:300])

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Agents as pure transforms | CrewAI agents don't have to be decision-makers |
| Tools do the data work | `parse_raw_data` + `calculate_scores` are deterministic |
| LLM only in final formatting | LLM overhead only where narrative matters |

### Next: [ADK Pipeline →](../ADK/pipeline.ipynb)